In [ ]:
import os, time, copy, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from torchvision.models import ResNet50_Weights
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, top_k_accuracy_score)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device   : {device}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_ROOT = '/content/drive/MyDrive/PlantLab2RealGeneralization'

SPLITS = {
    'Train'    : os.path.join(DATASET_ROOT, 'Train'),
    'Val'      : os.path.join(DATASET_ROOT, 'Val'),
    'Test_ID'  : os.path.join(DATASET_ROOT, 'Test_ID'),
    'Test_OOD' : os.path.join(DATASET_ROOT, 'Test_OOD'),
    'Few_Shot' : os.path.join(DATASET_ROOT, 'Few_Shot'),
}

print("Split verification:")
print(f"{'Split':<12} {'Exists':<8} {'Images':>8}")
print("-" * 30)
for name, path in SPLITS.items():
    exists = os.path.isdir(path)
    count  = sum(len(f) for _, _, f in os.walk(path)) if exists else 0
    status = "✓" if exists else "✗ NOT FOUND"
    print(f"  {name:<10} {status:<8} {count:>8,}")

Dataset Statistics & Class Distribution

Before training, inspecting the class distribution across all splits to understand:
- Whether classes are balanced (important for sampler choice)
- Whether `Test_OOD` has the same classes as `Train` (required for evaluation)
- The total dataset size

In [ ]:
def count_classes(folder):
    """Returns dict of {class_name: image_count}."""
    if not os.path.isdir(folder):
        return {}
    ds = datasets.ImageFolder(folder)
    counts = Counter(ds.targets)
    return {ds.classes[k]: v for k, v in sorted(counts.items())}

train_counts = count_classes(SPLITS['Train'])
ood_counts   = count_classes(SPLITS['Test_OOD'])

CLASS_NAMES = list(train_counts.keys())
NUM_CLASSES = len(CLASS_NAMES)

print(f"Number of classes : {NUM_CLASSES}")
print(f"\n{'Class':<45} {'Train':>7} {'OOD':>7}")
print("-" * 62)
for cls in CLASS_NAMES:
    t = train_counts.get(cls, 0)
    o = ood_counts.get(cls, 0)
    print(f"  {cls:<43} {t:>7,} {o:>7,}")

# Class distribution bar chart
fig, ax = plt.subplots(figsize=(14, 5))
x = range(NUM_CLASSES)
ax.bar(x, [train_counts.get(c, 0) for c in CLASS_NAMES],
       label='Train', color='steelblue', alpha=0.8)
ax.bar(x, [ood_counts.get(c,   0) for c in CLASS_NAMES],
       label='Test OOD', color='tomato', alpha=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=7)
ax.set_ylabel('Image Count'); ax.set_title('Class Distribution: Train vs Test_OOD')
ax.legend(); plt.tight_layout(); plt.show()

Compute Normalization Statistics

Normalization statistics (mean and std per channel) are computed **only from the
Train split**. Using validation or test data for normalization would be a form of
data leakage.

These values replace the default ImageNet statistics since PlantVillage has a very
different color distribution (predominantly green leaves on white backgrounds).

In [ ]:
def compute_mean_std(folder, img_size=224, batch_size=256, num_workers=4):
    """Compute per-channel mean and std from a dataset folder (Train only)."""
    t  = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])
    ds     = datasets.ImageFolder(folder, transform=t)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)

    mean = torch.zeros(3)
    std  = torch.zeros(3)
    n    = 0
    for imgs, _ in loader:
        b     = imgs.size(0) * imgs.size(2) * imgs.size(3)
        mean += imgs.mean(dim=[0, 2, 3]) * b
        std  += imgs.std(dim=[0, 2, 3])  * b
        n    += b

    mean /= n
    std  /= n
    return mean.tolist(), std.tolist()

print("Computing normalization statistics from Train split...")
NORM_MEAN, NORM_STD = compute_mean_std(SPLITS['Train'])

print(f"\n  Mean : {[round(m, 4) for m in NORM_MEAN]}")
print(f"  Std  : {[round(s, 4) for s in NORM_STD]}")
print("\nImageNet defaults for comparison:")
print(f"  Mean : [0.485, 0.456, 0.406]")
print(f"  Std  : [0.229, 0.224, 0.225]")

Data Transforms

Three transform pipelines are defined:

| Pipeline | Used For | Augmentations |
|---|---|---|
| `train_transform` | Train split | Heavy — targets the lab→field domain gap |
| `eval_transform` | Val, Test_ID, Test_OOD | Clean — resize + normalize only |
| `tta_transform` | Test-time augmentation | Light random flips + color jitter |

**Why heavy augmentation?** The domain gap between PlantVillage (white background, studio
lighting) and PlantDoc (real field images, varied backgrounds) is large. Simulating field
conditions during training, blur, perspective distortion, occlusion, and color variation,
encourages the model to learn features that transfer.

In [ ]:
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),         # zoom variation
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=45),
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.4, hue=0.15),                 # lighting variation
    transforms.RandomGrayscale(p=0.05),                               # overexposure sim
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),         # focus blur
    transforms.RandomPerspective(distortion_scale=0.3, p=0.3),        # angle variation
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),              # occlusion
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

tta_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

print("Transforms defined:")
print(f"  train_transform : {len(train_transform.transforms)} operations")
print(f"  eval_transform  : {len(eval_transform.transforms)} operations")
print(f"  tta_transform   : {len(tta_transform.transforms)} operations")

 Datasets & DataLoaders

A `WeightedRandomSampler` is used for the training loader to handle class imbalance,
PlantVillage has significantly more images for healthy leaves than for some disease
classes. This ensures each class is seen roughly equally during training.

> **Note:** `Few_Shot` does not get a loader here, it's only used if fine-tuning is
triggered in the final section.

In [ ]:
BATCH_SIZE   = 64
NUM_WORKERS  = 4

train_dataset    = datasets.ImageFolder(SPLITS['Train'],    transform=train_transform)
val_dataset      = datasets.ImageFolder(SPLITS['Val'],      transform=eval_transform)
test_id_dataset  = datasets.ImageFolder(SPLITS['Test_ID'],  transform=eval_transform)
test_ood_dataset = datasets.ImageFolder(SPLITS['Test_OOD'], transform=eval_transform)

# Sanity check: class names must match across splits
assert train_dataset.classes == val_dataset.classes, \
    "Class mismatch between Train and Val!"

CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f"{'Split':<12} {'Images':>8}  {'Classes':>8}")
print("-" * 32)
for name, ds in [('Train', train_dataset), ('Val', val_dataset),
                  ('Test_ID', test_id_dataset), ('Test_OOD', test_ood_dataset)]:
    print(f"  {name:<10} {len(ds):>8,}  {len(ds.classes):>8}")

def make_weighted_sampler(dataset):
    targets       = torch.tensor(dataset.targets)
    class_counts  = torch.bincount(targets)
    class_weights = 1.0 / class_counts.float()
    sample_wts    = class_weights[targets]
    return WeightedRandomSampler(sample_wts, len(sample_wts), replacement=True)

train_loader    = DataLoader(train_dataset,    batch_size=BATCH_SIZE,
                             sampler=make_weighted_sampler(train_dataset),
                             num_workers=NUM_WORKERS, pin_memory=True)
val_loader      = DataLoader(val_dataset,      batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
test_id_loader  = DataLoader(test_id_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
test_ood_loader = DataLoader(test_ood_dataset, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nDataLoaders ready  |  Batch size: {BATCH_SIZE}  |  Workers: {NUM_WORKERS}")

Model Architecture

**Base:** ResNet-50 pretrained on ImageNet (V2 weights — stronger than V1).

**Custom head** replaces the default single linear layer with a deeper classifier:

Dropout(0.5) → Linear(2048→512) → BatchNorm → ReLU → Dropout(0.3) → Linear(512→C)

**Why this head?** A 2-layer head with BatchNorm and two dropout stages is more
resistant to overfitting on PlantVillage and generalizes better to OOD data than
a single linear layer.

The backbone is **fully frozen** at initialization. Layers are progressively
unfrozen across 3 training phases.

In [ ]:
def build_model(num_classes: int, dropout: float = 0.5) -> nn.Module:
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    for param in model.parameters():
        param.requires_grad = False          # freeze entire backbone
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes),
    )
    return model

model     = build_model(NUM_CLASSES).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print(f"Model        : ResNet-50 (ImageNet V2 weights)")
print(f"Head         : Linear(2048→512→{NUM_CLASSES})")
print(f"Parameters   : {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)")

## 8. Training Utilities

This section defines all helper functions needed for training:

| Function | Purpose |
|---|---|
| `set_trainable_layers` | Freeze/unfreeze specific ResNet layer groups |
| `EarlyStopping` | Stop training when validation loss plateaus; saves best weights |
| `train_one_epoch` | One full pass over the training set with mixed precision |
| `evaluate_loader` | Compute loss and accuracy on any loader |
| `train_phase` | Run a full training phase with early stopping |
| `plot_history` | Plot loss and accuracy curves across all phases |

In [ ]:
def set_trainable_layers(model, layer_names: list):
    """Unfreeze named layer groups; freeze everything else."""
    for param in model.parameters():
        param.requires_grad = False
    for name in layer_names:
        for param in getattr(model, name).parameters():
            param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Unfrozen: {layer_names}  →  "
          f"{trainable:,} / {total:,} params ({100*trainable/total:.1f}%)")


class EarlyStopping:
    def __init__(self, patience: int = 8, min_delta: float = 1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = float('inf')
        self.best_state = None

    def step(self, val_loss: float, model: nn.Module) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter    = 0
            return False
        self.counter += 1
        return self.counter >= self.patience


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum, correct = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type):
            out  = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
    n = len(loader.dataset)
    return loss_sum / n, correct / n


@torch.no_grad()
def evaluate_loader(model, loader, criterion):
    model.eval()
    loss_sum, correct = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast(device_type=device.type):
            out  = model(imgs)
            loss = criterion(out, labels)
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
    n = len(loader.dataset)
    return loss_sum / n, correct / n


def train_phase(model, train_loader, val_loader, criterion, optimizer,
                scheduler, scaler, early_stopping, max_epochs, phase_name):
    print(f"\n{'='*65}\n  {phase_name}\n{'='*65}")
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion,
                                          optimizer, scaler)
        va_loss, va_acc = evaluate_loader(model, val_loader, criterion)

        if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(va_loss)
        else:
            scheduler.step()

        for k, v in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
            history[k].append(v)

        improved = va_loss <= early_stopping.best_loss
        tag = ' ✓ improved' if improved else \
              f' [{early_stopping.counter}/{early_stopping.patience}]'
        print(f"  Ep {epoch:3d}/{max_epochs}  "
              f"train {tr_loss:.4f}/{tr_acc:.4f}  "
              f"val {va_loss:.4f}/{va_acc:.4f}  "
              f"({time.time()-t0:.0f}s){tag}")

        if early_stopping.step(va_loss, model):
            print(f"\n  Early stopping triggered after {epoch} epochs.")
            break

    model.load_state_dict(early_stopping.best_state)
    print(f"  Best val loss: {early_stopping.best_loss:.4f}")
    return history


def plot_history(histories: dict):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    colors = ['#2196F3', '#4CAF50', '#FF9800']
    for i, (name, h) in enumerate(histories.items()):
        ep = range(1, len(h['train_loss']) + 1)
        c  = colors[i % len(colors)]
        ax1.plot(ep, h['train_loss'], '--', color=c, alpha=0.5, linewidth=1.2)
        ax1.plot(ep, h['val_loss'],   '-',  color=c, linewidth=2, label=name)
        ax2.plot(ep, h['train_acc'],  '--', color=c, alpha=0.5, linewidth=1.2)
        ax2.plot(ep, h['val_acc'],    '-',  color=c, linewidth=2, label=name)
    for ax, ylabel in [(ax1, 'Loss'), (ax2, 'Accuracy')]:
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)
        ax.set_title(f'{ylabel}  (solid = val, dashed = train)')
    plt.suptitle('Training Curves — All Phases', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()


print("All training utilities defined.")

Phase 1 — Train the FC Head Only

**Strategy:** Only the new classification head (`fc`) is trainable. The entire
ResNet-50 backbone remains frozen.

**Why start here?** Randomly initialized head weights can produce large gradients
that would damage pretrained backbone features if the backbone were unfrozen
immediately. Warming up the head first stabilizes training.

- **Optimizer:** AdamW (weight decay = 0.01)
- **LR:** 1e-3 (head only)  
- **Scheduler:** ReduceLROnPlateau (halves LR when val loss plateaus)
- **Max epochs:** 30 (early stopping with patience 8)

In [ ]:
criterion     = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler        = GradScaler(device_type=device.type)
all_histories = {}

set_trainable_layers(model, ['fc'])

opt_p1 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=0.01
)
sched_p1 = optim.lr_scheduler.ReduceLROnPlateau(
    opt_p1, mode='min', factor=0.5, patience=3, verbose=True
)
es_p1 = EarlyStopping(patience=8)

all_histories['Phase 1 (FC)'] = train_phase(
    model, train_loader, val_loader, criterion,
    opt_p1, sched_p1, scaler, es_p1,
    max_epochs=30, phase_name='Phase 1 — FC Head Only'
)

Phase 2 — Unfreeze layer4

**Strategy:** Unfreeze the last residual block (`layer4`) alongside the head.
These are the highest-level semantic features in ResNet — adapting them to
plant disease patterns is crucial for good transfer.

**Differential learning rates** are used to prevent overwriting the pretrained
features too aggressively:

| Layer group | LR |
|---|---|
| `layer4` | 1e-4 (conservative) |
| `fc` | 5e-4 (faster, already warmed up) |

- **Scheduler:** CosineAnnealingWarmRestarts (smooth LR cycling)
- **Max epochs:** 25

In [ ]:
set_trainable_layers(model, ['layer4', 'fc'])

opt_p2 = optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(),     'lr': 5e-4},
], weight_decay=0.01)

sched_p2 = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt_p2, T_0=10, T_mult=2, eta_min=1e-6
)
es_p2 = EarlyStopping(patience=8)

all_histories['Phase 2 (layer4)'] = train_phase(
    model, train_loader, val_loader, criterion,
    opt_p2, sched_p2, scaler, es_p2,
    max_epochs=25, phase_name='Phase 2 — Unfreeze layer4'
)

Phase 3 — Unfreeze layer3 + layer4

**Strategy:** Unfreeze mid-level features (`layer3`) for full fine-tuning of
the top half of ResNet. Three-level differential LR prevents catastrophic
forgetting of low-level texture features (still in frozen layer1/layer2).

| Layer group | LR |
|---|---|
| `layer3` | 1e-5 (very conservative) |
| `layer4` | 5e-5 |
| `fc` | 2e-4 |

- **Scheduler:** CosineAnnealingWarmRestarts
- **Max epochs:** 20 (tighter patience = 6 to avoid overfit)

After this phase, training curves for all 3 phases are plotted.

In [ ]:
set_trainable_layers(model, ['layer3', 'layer4', 'fc'])

opt_p3 = optim.AdamW([
    {'params': model.layer3.parameters(), 'lr': 1e-5},
    {'params': model.layer4.parameters(), 'lr': 5e-5},
    {'params': model.fc.parameters(),     'lr': 2e-4},
], weight_decay=0.01)

sched_p3 = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt_p3, T_0=10, T_mult=2, eta_min=1e-7
)
es_p3 = EarlyStopping(patience=6)

all_histories['Phase 3 (layer3+4)'] = train_phase(
    model, train_loader, val_loader, criterion,
    opt_p3, sched_p3, scaler, es_p3,
    max_epochs=20, phase_name='Phase 3 — Unfreeze layer3+4'
)

# Plot all phases together
plot_history(all_histories)

Evaluation Functions

Two evaluation modes are implemented:

**Standard evaluation** — single forward pass per image, fast.

**Test-Time Augmentation (TTA × 7)** — each test image is augmented 7 times and
predictions are averaged. This reduces variance and typically improves OOD accuracy
by 2–5% because it simulates the range of real-world image variations.

Results include:
- Top-1 and Top-5 accuracy
- Full per-class classification report
- Confusion matrix heatmap

In [ ]:
@torch.no_grad()
def full_evaluation(model, loader, class_names, split_name='Test',
                    use_tta=False, tta_n=7):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    if use_tta:
        ds = loader.dataset
        for img_path, label in ds.imgs:
            img   = ds.loader(img_path)
            probs = torch.zeros(len(class_names), device=device)
            for _ in range(tta_n):
                t = tta_transform(img).unsqueeze(0).to(device)
                with autocast(device_type=device.type):
                    probs += torch.softmax(model(t).squeeze(), dim=0)
            probs /= tta_n
            all_probs.append(probs.cpu())
            all_preds.append(probs.argmax().item())
            all_labels.append(label)
    else:
        for imgs, labels in loader:
            imgs = imgs.to(device)
            with autocast(device_type=device.type):
                out   = model(imgs)
                probs = torch.softmax(out, dim=1)
            all_probs.extend(probs.cpu().tolist())
            all_preds.extend(out.argmax(1).cpu().tolist())
            all_labels.extend(labels.tolist())

    probs_np = np.array([p.numpy() if hasattr(p, 'numpy') else p for p in all_probs])
    top1 = accuracy_score(all_labels, all_preds)
    top5 = top_k_accuracy_score(all_labels, probs_np,
                                 k=min(5, len(class_names)),
                                 labels=list(range(len(class_names))))
    tag = f' (TTA ×{tta_n})' if use_tta else ''
    print(f"\n{'='*60}")
    print(f"  {split_name}{tag}")
    print(f"{'='*60}")
    print(f"  Top-1 Accuracy : {top1:.4f}  ({100*top1:.2f}%)")
    print(f"  Top-5 Accuracy : {top5:.4f}  ({100*top5:.2f}%)")
    print(f"\n{classification_report(all_labels, all_preds, target_names=class_names, digits=4)}")

    # Confusion matrix
    cm  = confusion_matrix(all_labels, all_preds)
    sz  = max(10, NUM_CLASSES // 2)
    fig, ax = plt.subplots(figsize=(sz, sz))
    sns.heatmap(cm, annot=(NUM_CLASSES <= 20), fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)
    ax.set_title(f'Confusion Matrix — {split_name}{tag}', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=7)
    plt.yticks(rotation=0, fontsize=7)
    plt.tight_layout(); plt.show()

    return {
        'top1': top1, 'top5': top5,
        'preds': all_preds, 'labels': all_labels, 'probs': all_probs
    }

print("Evaluation functions defined.")

Evaluations

Three evaluations are run in sequence:

1. **Test_ID** — In-distribution (PlantVillage held-out). Measures the model's
   best-case performance on data from the same distribution as training.

2. **Test_OOD** — Out-of-distribution (PlantDoc + field data). This is the **key
   metric** for the project — measures real-world generalization.

3. **Test_OOD + TTA** — Same OOD data but with Test-Time Augmentation to boost
   robustness.

In [ ]:
results_id      = full_evaluation(model, test_id_loader,  CLASS_NAMES,
                                   'Test-ID  (PlantVillage in-dist)')
results_ood     = full_evaluation(model, test_ood_loader,  CLASS_NAMES,
                                   'Test-OOD (PlantDoc out-of-dist)')
results_ood_tta = full_evaluation(model, test_ood_loader,  CLASS_NAMES,
                                   'Test-OOD', use_tta=True, tta_n=7)

Domain Generalization Summary

This is the headline result of the experiment. The **domain gap** is defined as:


Domain Gap = Test_ID Accuracy − Test_OOD Accuracy

In [ ]:
gap      = results_id['top1'] - results_ood['top1']
tta_gain = results_ood_tta['top1'] - results_ood['top1']

print("=" * 55)
print("  DOMAIN GENERALIZATION SUMMARY")
print("=" * 55)
print(f"  Test-ID  (in-distribution)  : {results_id['top1']:.4f}  ({100*results_id['top1']:.2f}%)")
print(f"  Test-OOD (out-of-dist)      : {results_ood['top1']:.4f}  ({100*results_ood['top1']:.2f}%)")
print(f"  Test-OOD + TTA  (×7)        : {results_ood_tta['top1']:.4f}  ({100*results_ood_tta['top1']:.2f}%)")
print(f"  TTA gain                    : +{tta_gain:.4f}  (+{100*tta_gain:.2f}%)")
print(f"  Domain gap (ID − OOD)       : {gap:.4f}  ({100*gap:.2f}%)")
print()

if gap > 0.25:
    print("  ⚠️  LARGE gap — few-shot fine-tuning is strongly recommended.")
elif gap > 0.10:
    print("  ⚡  MODERATE gap — partial generalization achieved.")
else:
    print("  ✅  SMALL gap — strong generalization to real-world images!")

Save Checkpoint

The model checkpoint is saved with all metadata embedded — class names, normalization
statistics, and evaluation results — so it can be loaded and deployed without
needing any other files.

In [ ]:
SAVE_DIR = os.path.join(DATASET_ROOT, 'checkpoints')
os.makedirs(SAVE_DIR, exist_ok=True)

checkpoint = {
    'model_state_dict' : model.state_dict(),
    'class_names'      : CLASS_NAMES,
    'num_classes'      : NUM_CLASSES,
    'norm_mean'        : NORM_MEAN,
    'norm_std'         : NORM_STD,
    'img_size'         : IMG_SIZE,
    'results': {
        'test_id'      : {'top1': results_id['top1'],      'top5': results_id['top5']},
        'test_ood'     : {'top1': results_ood['top1'],     'top5': results_ood['top5']},
        'test_ood_tta' : {'top1': results_ood_tta['top1'], 'top5': results_ood_tta['top5']},
        'domain_gap'   : gap,
    }
}

save_path = os.path.join(SAVE_DIR, 'resnet50_plantlab2real.pth')
torch.save(checkpoint, save_path)
print(f"  Checkpoint saved to: {save_path}")
print(f"  File size          : {os.path.getsize(save_path) / 1e6:.1f} MB")

Few-Shot Fine-Tuning

Activates automatically if the domain gap exceeds **20%**.

**Strategy:** Fine-tune only `layer4` + `fc` on the field images at very low LRs
to adapt the model to real-world visual characteristics without catastrophic
forgetting of the PlantVillage features.

> Set `FORCE_FEW_SHOT = True` to run regardless of the domain gap.

In [ ]:
FORCE_FEW_SHOT = False
RUN_FEW_SHOT   = FORCE_FEW_SHOT or (gap > 0.20)

if RUN_FEW_SHOT:
    print(f"  Domain gap = {gap:.4f} → Running few-shot fine-tuning on Few_Shot split...\n")

    few_shot_dataset = datasets.ImageFolder(SPLITS['Few_Shot'],
                                             transform=train_transform)
    few_shot_loader  = DataLoader(few_shot_dataset, batch_size=32, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True)
    print(f"  Few_Shot images : {len(few_shot_dataset):,}")

    set_trainable_layers(model, ['layer4', 'fc'])

    opt_fs = optim.AdamW([
        {'params': model.layer4.parameters(), 'lr': 5e-5},
        {'params': model.fc.parameters(),     'lr': 2e-4},
    ], weight_decay=0.01)
    sched_fs = optim.lr_scheduler.CosineAnnealingLR(opt_fs, T_max=15, eta_min=1e-7)
    es_fs    = EarlyStopping(patience=5)

    all_histories['Few-Shot FT'] = train_phase(
        model, few_shot_loader, test_ood_loader, criterion,
        opt_fs, sched_fs, scaler, es_fs,
        max_epochs=15, phase_name='Few-Shot Fine-Tuning on Field Images'
    )
    plot_history({'Few-Shot FT': all_histories['Few-Shot FT']})

    # Re-evaluate OOD after fine-tuning
    results_ood_ft = full_evaluation(model, test_ood_loader, CLASS_NAMES,
                                      'Test-OOD (after few-shot FT)')
    improvement = results_ood_ft['top1'] - results_ood['top1']
    print(f"\n  OOD accuracy: {results_ood['top1']:.4f} → {results_ood_ft['top1']:.4f}  "
          f"({'+'if improvement>=0 else ''}{improvement:.4f})")

    # Save fine-tuned checkpoint
    ft_path = os.path.join(SAVE_DIR, 'resnet50_plantlab2real_finetuned.pth')
    torch.save({**checkpoint, 'model_state_dict': model.state_dict(),
                'few_shot_result': results_ood_ft['top1']}, ft_path)
    print(f"  Fine-tuned checkpoint saved: {ft_path}")
else:
    print(f"  Domain gap = {gap:.4f} ≤ 0.20 → Few-shot fine-tuning skipped.")
    print("  Set FORCE_FEW_SHOT = True to run anyway.")